<h3>구조 청킹과 시맨틱 청킹</h3>
<p><strong>개념</strong> 구조 청킹은 제목과 빈 줄 같은 문서 표식을 사용합니다. 시맨틱 청킹은 인접 문장의 임베딩 유사도가 낮아지는 위치를 경계 후보로 사용합니다.</p>

In [1]:
%pip install -q "python-dotenv==1.2.3" "openai==2.54.0"


Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
from math import sqrt
import re

load_dotenv(find_dotenv(usecwd=True))
client = OpenAI()
text = "# 배송\n결제 다음 날 출고합니다. 제주 지역은 이틀이 더 걸립니다. # 환불\n수령 후 7일 이내 신청합니다."
sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+|(?=# )", text) if s.strip()]


<h3>실제 임베딩으로 경계 찾기</h3>
<p><strong>확인할 점</strong> 이 셀은 <code>text-embedding-3-small</code>을 직접 호출합니다. 다섯 문장 이하의 작은 입력만 전송합니다.</p>

In [3]:
response = client.embeddings.create(
    model="text-embedding-3-small", input=sentences, dimensions=256
)
vectors = [item.embedding for item in response.data]
def cosine(a: list[float], b: list[float]) -> float:
    return sum(x*y for x, y in zip(a, b)) / (sqrt(sum(x*x for x in a))*sqrt(sum(y*y for y in b)))
scores = [round(cosine(a, b), 3) for a, b in zip(vectors, vectors[1:])]
boundary_threshold = 0.25
chunks, start = [], 0
for boundary in [i + 1 for i, score in enumerate(scores) if score < boundary_threshold]:
    chunks.append(" ".join(sentences[start:boundary])); start = boundary
chunks.append(" ".join(sentences[start:]))
print("sentences:", sentences)
print("adjacent cosine:", scores, "chunks:", chunks, "usage_tokens:", response.usage.total_tokens)


sentences: ['# 배송\n결제 다음 날 출고합니다.', '제주 지역은 이틀이 더 걸립니다.', '# 환불\n수령 후 7일 이내 신청합니다.']
adjacent cosine: [0.244, 0.269] chunks: ['# 배송\n결제 다음 날 출고합니다.', '제주 지역은 이틀이 더 걸립니다. # 환불\n수령 후 7일 이내 신청합니다.'] usage_tokens: 57


<h3>일부러 실패해 보기</h3>
<p><strong>실패 사례</strong> 임계값을 지나치게 높이면 거의 모든 문장 사이가 경계가 됩니다. 다음 셀은 이 과분할을 포착해 보여 줍니다.</p>

In [4]:
try:
    threshold = 0.999
    boundaries = [i + 1 for i, score in enumerate(scores) if score < threshold]
    assert len(boundaries) < len(sentences) - 1, "거의 모든 문장이 분리되었습니다"
except AssertionError as error:
    print(f"caught: {error}; boundaries={boundaries}")


caught: 거의 모든 문장이 분리되었습니다; boundaries=[1, 2]
